## 1. Setup
Install dependencies and confirm GPU availability.

In [ ]:
import os

RESULTS_FILE = "/kaggle/working/results.csv"

if os.path.exists(RESULTS_FILE):
    results_table = pd.read_csv(RESULTS_FILE).to_dict("records")
    print("Loaded existing results:", [r["model"] for r in results_table])
else:
    results_table = []

In [1]:
!pip install -q jiwer
# transformers, torch, and datasets are pre-installed on Kaggle, but this keeps them current
!pip install -q -U transformers datasets accelerate

In [2]:
import torch
print(torch.cuda.is_available())  # should print True if GPU is on

True


## 2. Data
Download the Basque Common Voice corpus (v27) via the Mozilla Data Collective API, then extract only the test split we need (test.tsv + matching audio clips) rather than the full 14GB archive.

In [3]:
import requests
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("MOZILLA")

response = requests.post(
    "https://mozilladatacollective.com/api/datasets/cmu5xc07l00e6o10740qugb6a/download",
    headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
)
download_url = response.json()["downloadUrl"]

with requests.get(download_url, stream=True) as r:
    with open("basque-common-voice.tar.gz", "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

print("Download complete")

Download complete


In [4]:
import tarfile

# ONLY RUN IF NEEDING TO KNOW FOLDER STRUCTURE
with tarfile.open("basque-common-voice.tar.gz", "r:gz") as tar:
    names = tar.getnames()
    print(names[:20])  # first 20 entries, to see the folder structure

['cv-corpus-27.0-2026-09-11/eu/README.md', 'cv-corpus-27.0-2026-09-11/eu/clip_durations.tsv', 'cv-corpus-27.0-2026-09-11/eu/clips', 'cv-corpus-27.0-2026-09-11/eu/dev.tsv', 'cv-corpus-27.0-2026-09-11/eu/invalidated.tsv', 'cv-corpus-27.0-2026-09-11/eu/other.tsv', 'cv-corpus-27.0-2026-09-11/eu/reported.tsv', 'cv-corpus-27.0-2026-09-11/eu/test.tsv', 'cv-corpus-27.0-2026-09-11/eu/train.tsv', 'cv-corpus-27.0-2026-09-11/eu/unvalidated_sentences.tsv', 'cv-corpus-27.0-2026-09-11/eu/validated.tsv', 'cv-corpus-27.0-2026-09-11/eu/validated_sentences.tsv', 'cv-corpus-27.0-2026-09-11/eu/clips/common_voice_eu_18251595.mp3', 'cv-corpus-27.0-2026-09-11/eu/clips/common_voice_eu_18251596.mp3', 'cv-corpus-27.0-2026-09-11/eu/clips/common_voice_eu_18251597.mp3', 'cv-corpus-27.0-2026-09-11/eu/clips/common_voice_eu_18251598.mp3', 'cv-corpus-27.0-2026-09-11/eu/clips/common_voice_eu_18251599.mp3', 'cv-corpus-27.0-2026-09-11/eu/clips/common_voice_eu_18251640.mp3', 'cv-corpus-27.0-2026-09-11/eu/clips/common_voice

In [5]:
with tarfile.open("basque-common-voice.tar.gz", "r:gz") as tar:
    tar.extract("cv-corpus-27.0-2026-09-11/eu/test.tsv", path=".")

print("test.tsv extracted")

/tmp/ipykernel_5134/3239964304.py:2: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract("cv-corpus-27.0-2026-09-11/eu/test.tsv", path=".")


test.tsv extracted


In [6]:
import pandas as pd

test_df = pd.read_csv("cv-corpus-27.0-2026-09-11/eu/test.tsv", sep="\t")
print(test_df.shape)
print(test_df[["path", "sentence"]].head())

SAMPLE_SIZE = len(test_df)  # None/full set; change to e.g. 2000 for a smaller test run

(14809, 13)
                           path  \
0  common_voice_eu_22343411.mp3   
1  common_voice_eu_39176289.mp3   
2  common_voice_eu_39176288.mp3   
3  common_voice_eu_39176290.mp3   
4  common_voice_eu_39176287.mp3   

                                            sentence  
0  Egun honek Latinoamerikan eta Karibean du jato...  
1  Egunaren amaieran txingorra egin lezake, eta t...  
2                  Emakumeak ihes egitea lortu zuen.  
3                  Ez da izan ikasi dudan gauza bat.  
4  Horrelako jarrerak mugatzeko neurriak hartu di...  


In [7]:
# TO DETERMINE COLUMNS IN TEST DATAFRAME
print(test_df.columns.tolist())

['client_id', 'path', 'sentence_id', 'sentence', 'sentence_domain', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant', 'locale', 'segment']


In [8]:
clip_names = set(test_df["path"])  # fast lookup
prefix = "cv-corpus-27.0-2026-09-11/eu/clips/"

with tarfile.open("basque-common-voice.tar.gz", "r:gz") as tar:
    for member in tar:
        if member.name.startswith(prefix) and member.name[len(prefix):] in clip_names:
            tar.extract(member, path=".")

print("Test clips extracted")

/tmp/ipykernel_5134/3986926503.py:7: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path=".")


Test clips extracted


In [9]:
import os
clips_path = "cv-corpus-27.0-2026-09-11/eu/clips"
print(len(os.listdir(clips_path)))

14809


## 3. Utilities
Shared helper functions used by every model below: a timing wrapper (for RTF) and the WER/CER/error-breakdown scoring functions.

In [10]:
import time
import librosa


def transcribe_with_timing(pipe, audio_path):
    duration = librosa.get_duration(path=audio_path)
    start = time.time()
    text = pipe(audio_path)["text"]
    elapsed = time.time() - start
    rtf = elapsed / duration
    return text, rtf


def transcribe_with_timing_sensevoice(model, audio_path):
    duration = librosa.get_duration(path=audio_path)
    start = time.time()
    result = model.generate(input=audio_path, language="auto")
    text = result
    elapsed = time.time() - start
    rtf = elapsed / duration
    return text, rtf

In [11]:
# Normalise SenseVoice-Small text output
import re

def clean_sensevoice_output(text):
    return re.sub(r"<\|.*?\|>", "", text).strip()

In [12]:
def transcribe_with_timing_sensevoice(model, audio_path):
    duration = librosa.get_duration(path=audio_path)
    start = time.time()
    result = model.generate(input=audio_path, language="auto")
    text = clean_sensevoice_output(result[0]["text"])
    elapsed = time.time() - start
    rtf = elapsed / duration
    return text, rtf

In [13]:
import jiwer


def compute_wer(predictions, references):
    # normalize: lowercase, strip punctuation, so formatting differences don't count as errors
    transform = jiwer.Compose([
        jiwer.ToLowerCase(),
        jiwer.RemovePunctuation(),
        jiwer.Strip(),
    ])
    predictions = [transform(p) for p in predictions]
    references = [transform(r) for r in references]
    
    return jiwer.wer(references, predictions)


def compute_cer(predictions, references):
    transform = jiwer.Compose([
        jiwer.ToLowerCase(),
        jiwer.RemovePunctuation(),
        jiwer.Strip(),
    ])
    predictions = [transform(p) for p in predictions]
    references = [transform(r) for r in references]
    return jiwer.cer(references, predictions)


def compute_error_breakdown(predictions, references):
    transform = jiwer.Compose([
        jiwer.ToLowerCase(),
        jiwer.RemovePunctuation(),
        jiwer.Strip(),
    ])
    predictions = [transform(p) for p in predictions]
    references = [transform(r) for r in references]
    output = jiwer.process_words(references, predictions)
    return {
        "substitutions": output.substitutions,
        "insertions": output.insertions,
        "deletions": output.deletions,
    }

In [14]:
# Utility for a results table to collate model metrics
results_table = []

import os

def add_model_result(name, wer, cer, breakdown, rtfs, basque_claimed):
    results_table.append({
        "model": name,
        "WER": round(wer, 4),
        "CER": round(cer, 4),
        "substitutions": breakdown["substitutions"],
        "insertions": breakdown["insertions"],
        "deletions": breakdown["deletions"],
        "avg_RTF": round(sum(rtfs) / len(rtfs), 4),
        "basque_claimed": basque_claimed,
    })
    pd.DataFrame(results_table).to_csv(RESULTS_FILE, index=False)
    print(f"Saved results for {name} to {RESULTS_FILE}")

import pandas as pd
def show_results_table():
    return pd.DataFrame(results_table)

### Authenticate with HuggingFace
Needed for higher rate limits / faster model downloads.

In [15]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret("hf_basque_benchmarking")
login(token=hf_token)

## 4.1 Model: Whisper-large-v3
Load the model and build a transcription pipeline.

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device == "cuda" else torch.float32

from transformers import pipeline
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

model_id = "openai/whisper-large-v3"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

print("Model loaded")

asr_pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
)

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

In [ ]:
# Run inference + timing on the first n test clips (small sample to validate before scaling to all 14,801)
sample_paths = [f"{clips_path}/{name}" for name in test_df["path"].head(SAMPLE_SIZE)]
results = [transcribe_with_timing(asr_pipe, p) for p in sample_paths]
transcripts = [r[0] for r in results]
rtfs = [r[1] for r in results]

print(transcripts)
print(rtfs)

In [ ]:
ground_truth = test_df["sentence"].head(SAMPLE_SIZE).tolist()

wer = compute_wer(transcripts, ground_truth)
cer = compute_cer(transcripts, ground_truth)
breakdown = compute_error_breakdown(transcripts, ground_truth)

print(f"WER: {wer}")
print(f"CER: {cer}")
print(breakdown)

add_model_result("Whisper-large-v3", wer, cer, breakdown, rtfs, basque_claimed=True)

## 4.2 Model: Whisper-tiny-eu
Load the fine-tuned Basque model and build a transcription pipeline.

In [ ]:
# Load whisper-tiny-eu
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device == "cuda" else torch.float32

model_id_tiny = "zuazo/whisper-tiny-eu"

model_tiny = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id_tiny, dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model_tiny.to(device)

processor_tiny = AutoProcessor.from_pretrained(model_id_tiny)

asr_pipe_tiny = pipeline(
    "automatic-speech-recognition",
    model=model_tiny,
    tokenizer=processor_tiny.tokenizer,
    feature_extractor=processor_tiny.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
)

print("Whisper-tiny-eu loaded")

In [20]:
# Score whisper-tiny-eu
sample_paths = [f"{clips_path}/{name}" for name in test_df["path"].head(SAMPLE_SIZE)]
results_tiny = [transcribe_with_timing(asr_pipe_tiny, p) for p in sample_paths]
transcripts_tiny = [r[0] for r in results_tiny]
rtfs_tiny = [r[1] for r in results_tiny]

ground_truth = test_df["sentence"].head(SAMPLE_SIZE).tolist()

wer_tiny = compute_wer(transcripts_tiny, ground_truth)
cer_tiny = compute_cer(transcripts_tiny, ground_truth)
breakdown_tiny = compute_error_breakdown(transcripts_tiny, ground_truth)

print(f"WER: {wer_tiny}")
print(f"CER: {cer_tiny}")
print(breakdown_tiny)
print(f"Average RTF: {sum(rtfs_tiny)/len(rtfs_tiny)}")

add_model_result("Whisper-tiny-eu", wer_tiny, cer_tiny, breakdown_tiny, rtfs_tiny, basque_claimed=True)

WER: 0.306698746359377
CER: 0.07113617054143817
{'substitutions': 3750, 'insertions': 611, 'deletions': 483}
Average RTF: 0.0629812476102404


## 4.3 Model: SenseVoice-Large
Load the fine-tuned Basque model and build a transcription pipeline.

In [21]:
# Install funasr first (one-time)
!pip install -q funasr

from funasr import AutoModel

model_svl = AutoModel(model="iic/SenseVoiceSmall", device=device)
print("SenseVoice-Small loaded")

test_result = model_svl.generate(input=f"{clips_path}/{test_df['path'].iloc[0]}", language="auto")
print(test_result)

2026-10-06 20:24:43,500 [INFO] download models from model hub: ms


funasr version: 1.4.16.
Check update of funasr, and it would cost few times. You may disable it by set `disable_update=True` in AutoModel
You are using the latest version of funasr-1.4.16


2026-10-06 20:24:45,091 | INFO    | modelscope_hub.download | Downloading 20 files from iic/SenseVoiceSmall@master


Downloading:   0%|          | 0/20 [00:00<?, ?file/s]

2026-10-06 20:24:48,676 [WARNING] trust_remote_code: False
2026-10-06 20:24:50,546 [INFO] Loading pretrained params from /root/.cache/modelscope/models/iic--SenseVoiceSmall/snapshots/master/model.pt
2026-10-06 20:24:50,555 [INFO] ckpt: /root/.cache/modelscope/models/iic--SenseVoiceSmall/snapshots/master/model.pt
2026-10-06 20:24:51,787 [INFO] scope_map: ['module.', 'None']
2026-10-06 20:24:51,788 [INFO] excludes: None
2026-10-06 20:24:51,903 [INFO] Loading ckpt: /root/.cache/modelscope/models/iic--SenseVoiceSmall/snapshots/master/model.pt, status: <All keys matched successfully>


SenseVoice-Small loaded


rtf_avg: 0.079: 100%|██████████| 1/1 [00:00<00:00,  2.66it/s]                                                                                          

[{'key': 'common_voice_eu_22343411', 'text': '<|en|><|NEUTRAL|><|Speech|><|woitn|>ehuon ne latin america nettaib and du hatorian'}]


In [ ]:
sample_paths = [f"{clips_path}/{name}" for name in test_df["path"].head(SAMPLE_SIZE)]
results_svl = [transcribe_with_timing_sensevoice(model_svl, p) for p in sample_paths]
transcripts_svl = [r[0] for r in results_svl]
rtfs_svl = [r[1] for r in results_svl]

ground_truth = test_df["sentence"].head(SAMPLE_SIZE).tolist()

wer_svl = compute_wer(transcripts_svl, ground_truth)
cer_svl = compute_cer(transcripts_svl, ground_truth)
breakdown_svl = compute_error_breakdown(transcripts_svl, ground_truth)

print(f"WER: {wer_svl}")
print(f"CER: {cer_svl}")
print(breakdown_svl)

add_model_result("SenseVoice-Small", wer_svl, cer_svl, breakdown_svl, rtfs_svl, basque_claimed=False)

rtf_avg: 0.017: 100%|██████████| 1/1 [00:00<00:00, 11.23it/s]                                                                                          


## 4.4 Model: alexdimmock/wav2vec2-basque-200h
Load the fine-tuned Basque model and build a transcription pipeline.

In [ ]:
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

model_id_w2v = "alexdimmock/wav2vec2-basque-200h"  # confirm this matches your actual pushed repo name on HF

model_w2v = Wav2Vec2ForCTC.from_pretrained(model_id_w2v)
model_w2v.to(device)

processor_w2v = Wav2Vec2Processor.from_pretrained(model_id_w2v)

asr_pipe_w2v = pipeline(
    "automatic-speech-recognition",
    model=model_w2v,
    tokenizer=processor_w2v.tokenizer,
    feature_extractor=processor_w2v.feature_extractor,
    device=device,
)

print("wav2vec2-Basque loaded")

In [ ]:
sample_paths_w2v = [f"{clips_path}/{name}" for name in test_df["path"].head(SAMPLE_SIZE)]
results_w2v = [transcribe_with_timing(asr_pipe_w2v, p) for p in sample_paths_w2v]
transcripts_w2v = [r[0] for r in results_w2v]
rtfs_w2v = [r[1] for r in results_w2v]

ground_truth = test_df["sentence"].head(SAMPLE_SIZE).tolist()

wer_w2v = compute_wer(transcripts_w2v, ground_truth)
cer_w2v = compute_cer(transcripts_w2v, ground_truth)
breakdown_w2v = compute_error_breakdown(transcripts_w2v, ground_truth)

print(f"WER: {wer_w2v}")
print(f"CER: {cer_w2v}")
print(breakdown_w2v)

add_model_result("wav2vec2-Basque (own)", wer_w2v, cer_w2v, breakdown_w2v, rtfs_w2v, basque_claimed=True)

## 5. Results

In [ ]:
# Print collated model metrics table
show_results_table()